In [1]:
import duckdb
con = duckdb.connect("olist.duckdb")
print("connected")

connected


In [2]:
con.execute("""
CREATE OR REPLACE TABLE delivery_orders AS
WITH d AS (
  SELECT order_id, customer_state, review_score,
         DATE_DIFF('day', CAST(order_estimated_delivery_date AS DATE),
                          CAST(order_delivered_customer_date AS DATE)) AS days_late,
         DATE_DIFF('day', CAST(order_purchase_timestamp AS DATE),
                          CAST(order_delivered_customer_date AS DATE)) AS delivery_days,
         DATE_DIFF('day', CAST(order_purchase_timestamp AS DATE),
                          CAST(order_estimated_delivery_date AS DATE)) AS estimated_days
  FROM order_level
  WHERE order_status = 'delivered'
    AND order_delivered_customer_date IS NOT NULL
)
SELECT *,
       CASE WHEN days_late > 0 THEN 'Late' ELSE 'On time' END AS delivery_status,
       CASE WHEN days_late <= 0  THEN 'On time or early'
            WHEN days_late <= 3  THEN '1-3 days late'
            WHEN days_late <= 7  THEN '4-7 days late'
            WHEN days_late <= 14 THEN '8-14 days late'
            ELSE '15+ days late' END AS lateness_band,
       CASE WHEN days_late <= 0  THEN 0
            WHEN days_late <= 3  THEN 1
            WHEN days_late <= 7  THEN 2
            WHEN days_late <= 14 THEN 3
            ELSE 4 END AS band_order
FROM d
""")

con.execute("SELECT * FROM delivery_orders LIMIT 5").df()

,order_id,customer_state,review_score,days_late,delivery_days,estimated_days,delivery_status,lateness_band,band_order
0,e481f51cbdc54678b7cc49136f2d6af7,SP,4.0,-8,8,16,On time,On time or early,0
1,53cdb2fc8bc7dce0b6741e2150273451,BA,4.0,-6,14,20,On time,On time or early,0
2,47770eb9100c2d0c44946d9cf07ec65d,GO,5.0,-18,9,27,On time,On time or early,0
3,ad21c59c0840e6cb83a9ceb5573f8159,SP,5.0,-10,3,13,On time,On time or early,0
4,a4591c265e18cb1dcee52889e2d8acc3,PR,4.0,-6,17,23,On time,On time or early,0


In [3]:
con.execute("""
CREATE OR REPLACE TABLE delivery_overall AS
SELECT COUNT(*) AS delivered_orders,
       SUM(CASE WHEN days_late > 0 THEN 1 ELSE 0 END) AS late_orders,
       ROUND(100.0 * SUM(CASE WHEN days_late > 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_pct,
       ROUND(AVG(CASE WHEN days_late > 0 THEN days_late END), 1) AS avg_days_late_when_late,
       ROUND(AVG(delivery_days), 1) AS avg_delivery_days,
       ROUND(AVG(estimated_days), 1) AS avg_estimated_days
FROM delivery_orders
""")

con.execute("SELECT * FROM delivery_overall").df()

,delivered_orders,late_orders,late_pct,avg_days_late_when_late,avg_delivery_days,avg_estimated_days
0,96470,6534.0,6.77,10.6,12.5,24.4


In [4]:
con.execute("""
CREATE OR REPLACE TABLE delivery_status_summary AS
SELECT delivery_status,
       COUNT(*) AS orders,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_orders,
       COUNT(review_score) AS reviewed_orders,
       ROUND(AVG(review_score), 2) AS avg_review,
       ROUND(100.0 * SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) / COUNT(review_score), 2) AS low_review_pct,
       ROUND(AVG(delivery_days), 1) AS avg_delivery_days
FROM delivery_orders
GROUP BY 1
ORDER BY 1
""")

con.execute("SELECT * FROM delivery_status_summary").df()

,delivery_status,orders,pct_of_orders,reviewed_orders,avg_review,low_review_pct,avg_delivery_days
0,Late,6534,6.77,6381,2.27,62.36,33.9
1,On time,89936,93.23,89443,4.29,9.23,10.9


In [5]:
con.execute("""
CREATE OR REPLACE TABLE delivery_by_band AS
SELECT lateness_band, band_order,
       COUNT(*) AS orders,
       ROUND(AVG(review_score), 2) AS avg_review,
       ROUND(100.0 * SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) / COUNT(review_score), 2) AS low_review_pct
FROM delivery_orders
GROUP BY 1, 2
ORDER BY band_order
""")

con.execute("SELECT * FROM delivery_by_band ORDER BY band_order").df()

,lateness_band,band_order,orders,avg_review,low_review_pct
0,On time or early,0,89936,4.29,9.23
1,1-3 days late,1,1870,3.29,32.18
2,4-7 days late,2,1802,2.11,67.56
3,8-14 days late,3,1478,1.67,80.08
4,15+ days late,4,1384,1.73,78.20


In [6]:
con.execute("""
CREATE OR REPLACE TABLE delivery_by_state AS
SELECT customer_state,
       COUNT(*) AS orders,
       SUM(CASE WHEN days_late > 0 THEN 1 ELSE 0 END) AS late_orders,
       ROUND(100.0 * SUM(CASE WHEN days_late > 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_pct,
       ROUND(AVG(delivery_days), 1) AS avg_delivery_days,
       ROUND(AVG(review_score), 2) AS avg_review
FROM delivery_orders
GROUP BY 1
HAVING COUNT(*) >= 100
ORDER BY late_pct DESC
""")

con.execute("SELECT * FROM delivery_by_state").df()

,customer_state,orders,late_orders,late_pct,avg_delivery_days,avg_review
0,AL,397,85.0,21.41,24.5,3.85
1,MA,717,125.0,17.43,21.5,3.83
2,SE,335,51.0,15.22,21.5,3.91
3,PI,476,66.0,13.87,19.4,3.99
4,CE,1279,176.0,13.76,21.2,3.94
5,BA,3256,396.0,12.16,19.3,3.93
6,RJ,12350,1495.0,12.11,15.2,3.97
7,PA,946,106.0,11.21,23.7,3.91
8,ES,1995,214.0,10.73,15.7,4.08
9,PB,517,54.0,10.44,20.4,4.08


In [7]:
con.execute("""
SELECT
  (SELECT COUNT(*) FROM order_level
   WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL) AS expected_orders,
  (SELECT COUNT(*) FROM delivery_orders) AS delivery_orders_rows,
  (SELECT COUNT(DISTINCT order_id) FROM delivery_orders) AS distinct_orders,
  (SELECT SUM(orders) FROM delivery_status_summary) AS status_summary_total,
  (SELECT SUM(orders) FROM delivery_by_band) AS band_total,
  (SELECT SUM(orders) FROM delivery_by_state) AS state_total
""").df()

,expected_orders,delivery_orders_rows,distinct_orders,status_summary_total,band_total,state_total
0,96470,96470,96470,96470.0,96470.0,96282.0


In [8]:
con.execute("COPY delivery_overall TO 'dashboard/delivery_overall.csv' (HEADER, DELIMITER ',')")
con.execute("COPY delivery_status_summary TO 'dashboard/delivery_status_summary.csv' (HEADER, DELIMITER ',')")
con.execute("COPY delivery_by_band TO 'dashboard/delivery_by_band.csv' (HEADER, DELIMITER ',')")
con.execute("COPY delivery_by_state TO 'dashboard/delivery_by_state.csv' (HEADER, DELIMITER ',')")
print("saved")

saved


In [9]:
o = con.execute("SELECT * FROM delivery_overall").df().iloc[0]
s = con.execute("SELECT * FROM delivery_status_summary").df().set_index("delivery_status")
b = con.execute("SELECT * FROM delivery_by_band ORDER BY band_order").df()
st = con.execute("SELECT * FROM delivery_by_state").df()

In [10]:
late, ontime = s.loc["Late"], s.loc["On time"]
gap = round(ontime.avg_review - late.avg_review, 2)
worst3 = st.nlargest(3, "late_pct")
best = st.nsmallest(1, "late_pct").iloc[0]
worst_txt = ", ".join(f"{r.customer_state} ({r.late_pct}%)" for r in worst3.itertuples())
last_band = b.iloc[-1]

In [11]:
print("=== COPY THIS INTO business_questions.md (Q5 answer) ===")
print(f"- Answer: Of {int(o.delivered_orders):,} delivered orders with a delivery date, {int(o.late_orders):,} ({o.late_pct}%) arrived after the estimated date, and late orders were on average {o.avg_days_late_when_late} days late. "
      f"Orders took {o.avg_delivery_days} days on average against {o.avg_estimated_days} days estimated. "
      f"Among states with at least 100 delivered orders, the highest late rates are {worst_txt}; the lowest is {best.customer_state} ({best.late_pct}%).")
print()
print("=== COPY THIS INTO business_questions.md (Q6 answer) ===")
print(f"- Answer: Late orders average a review score of {late.avg_review} against {ontime.avg_review} for on-time orders, a gap of {gap} points, and {late.low_review_pct}% of late orders have a 1-2 star review against {ontime.low_review_pct}% of on-time orders. "
      f"Orders delivered {last_band.lateness_band} average {last_band.avg_review}. "
      f"This shows a strong association between lateness and low reviews; it does not prove that lateness alone causes them.")
print()
print("=== COPY THIS INTO README.md (under ## Method) ===")
print("- Delivery analysis uses delivered orders that have a recorded delivery date. An order is late if its delivery date (date only, not time) is after the estimated delivery date; delivery on the estimated day counts as on time. State rankings use states with at least 100 delivered orders. Review comparisons show association, not proof of cause.")

=== COPY THIS INTO business_questions.md (Q5 answer) ===
- Answer: Of 96,470 delivered orders with a delivery date, 6,534 (6.77%) arrived after the estimated date, and late orders were on average 10.6 days late. Orders took 12.5 days on average against 24.4 days estimated. Among states with at least 100 delivered orders, the highest late rates are AL (21.41%), MA (17.43%), SE (15.22%); the lowest is AM (2.76%).

=== COPY THIS INTO business_questions.md (Q6 answer) ===
- Answer: Late orders average a review score of 2.27 against 4.29 for on-time orders, a gap of 2.02 points, and 62.36% of late orders have a 1-2 star review against 9.23% of on-time orders. Orders delivered 15+ days late average 1.73. This shows a strong association between lateness and low reviews; it does not prove that lateness alone causes them.

=== COPY THIS INTO README.md (under ## Method) ===
- Delivery analysis uses delivered orders that have a recorded delivery date. An order is late if its delivery date (date o